In [1]:
!pip install -q openai pandas numpy matplotlib seaborn scipy

In [25]:
from openai import OpenAI
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
import json
import re
import time
from scipy.stats import spearmanr, kendalltau

BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"  # Ollama accepts a placeholder key for the OpenAI-compatible API
GENERATOR_MODEL = "llama3.1:latest"
JUDGE_MODEL = "llama3.1:latest"  # Prefer a different judge model if available

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Quick connection test
response = client.chat.completions.create(
    model=GENERATOR_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    temperature=0,
    max_tokens=10
)
print(response.choices[0].message.content)

OK


In [26]:
GENERATION_PROMPT = """Generate a concise, natural message expressing the extent to which the viewpoint in the Original Post is reflected in the target user's response, consistent with the Target Profile and the given stance-change magnitude Δ.

Interpret Δ as a normalized control signal in the range [0, 1] indicating the strength of the viewpoint to be expressed:
- Δ close to 0: express little or no clear reflection of the Original Post's viewpoint. The message may acknowledge the topic without clearly endorsing the viewpoint.
- Moderate Δ: express a partial reflection of the viewpoint, agreeing with or incorporating some aspects without fully endorsing it.
- Δ close to 1: express the viewpoint more clearly and explicitly, while allowing reservations or selective agreement.

Intermediate values should produce intermediate levels of expression where appropriate. These are qualitative guidelines, not exact numerical mappings. Do not interpret Δ as a probability, a percentage of the message internalized, or a guarantee of behavioral change.

Be consistent with the Target Profile. Do not assume complete acceptance solely because Δ is large. Do not introduce unrelated claims or copy the Original Post verbatim. Output only the generated message in 1–3 sentences.

Original Post:
{post}

Target Profile:
{profile}

Stance Change Magnitude (Δ):
{delta}

Generate only the internalized message."""


def generate_internalized_message(post, profile, delta, temperature=0.7, max_tokens=140):
    prompt = GENERATION_PROMPT.format(
        post=post,
        profile=profile,
        delta=f"{float(delta):.2f}"
    )
    response = client.chat.completions.create(
        model=GENERATOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

In [ ]:
!python --version
!python -m pip install seaborn
!pip install seaborn

In [27]:
test_cases = [
    {
        "case_id": "policy_01",
        "post": "The city should expand protected bike lanes because they can make cycling safer and reduce dependence on cars.",
        "profile": "A local resident who supports environmental protection but worries about road congestion and parking."
    },
    {
        "case_id": "policy_02",
        "post": "The city should provide more funding for public libraries because they offer accessible learning spaces and community services.",
        "profile": "A budget-conscious resident who values education but is cautious about increasing public spending."
    },
]

DELTA_VALUES = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
N_REPEATS = 3  # Increase to 5–10 for a more stable exploratory evaluation
GENERATION_TEMPERATURE = 0.7

print(f"{len(test_cases)} cases × {len(DELTA_VALUES)} delta values × {N_REPEATS} repeats")

2 cases × 7 delta values × 3 repeats


In [28]:
generation_rows = []

for case in test_cases:
    for delta in DELTA_VALUES:
        for repeat in range(N_REPEATS):
            try:
                message = generate_internalized_message(
                    post=case["post"],
                    profile=case["profile"],
                    delta=delta,
                    temperature=GENERATION_TEMPERATURE
                )
                generation_rows.append({
                    "case_id": case["case_id"],
                    "post": case["post"],
                    "profile": case["profile"],
                    "delta": delta,
                    "repeat": repeat + 1,
                    "generated_message": message,
                    "generation_model": GENERATOR_MODEL,
                    "temperature": GENERATION_TEMPERATURE,
                    "error": None
                })
                print(f'{case["case_id"]} | Δ={delta:.1f} | run={repeat+1}: {message[:100]}')
            except Exception as e:
                generation_rows.append({
                    "case_id": case["case_id"],
                    "post": case["post"],
                    "profile": case["profile"],
                    "delta": delta,
                    "repeat": repeat + 1,
                    "generated_message": None,
                    "generation_model": GENERATOR_MODEL,
                    "temperature": GENERATION_TEMPERATURE,
                    "error": str(e)
                })
                print("ERROR:", case["case_id"], delta, repeat + 1, e)
            time.sleep(0.1)

generations = pd.DataFrame(generation_rows)
generations.to_csv("internalized_message_generations.csv", index=False, encoding="utf-8-sig")
generations.head()

policy_01 | Δ=0.0 | run=1: I'm not sure I agree that the city should focus solely on expanding protected bike lanes, as it may 
policy_01 | Δ=0.0 | run=2: "I'm aware of the proposal to expand protected bike lanes, and I'm open to considering its potential
policy_01 | Δ=0.0 | run=3: "I'm aware of the proposal to expand protected bike lanes, but I have concerns about how it would im
policy_01 | Δ=0.1 | run=1: I'm cautiously in favor of expanding protected bike lanes, as they have the potential to make cyclin
policy_01 | Δ=0.1 | run=2: While I appreciate the idea of expanding protected bike lanes, I'm still concerned about the potenti
policy_01 | Δ=0.1 | run=3: "I'm not entirely convinced that expanding protected bike lanes is the solution, but I do think it's
policy_01 | Δ=0.3 | run=1: While I agree that protected bike lanes can be a step in the right direction for the environment, I 
policy_01 | Δ=0.3 | run=2: "I understand the appeal of protected bike lanes, but I'm concerned that expa

,case_id,post,profile,delta,repeat,generated_message,generation_model,temperature,error
0,policy_01,The city should expand protected bike lanes be...,A local resident who supports environmental pr...,0.0,1,I'm not sure I agree that the city should focu...,llama3.1:latest,0.7,None
1,policy_01,The city should expand protected bike lanes be...,A local resident who supports environmental pr...,0.0,2,"""I'm aware of the proposal to expand protected...",llama3.1:latest,0.7,None
2,policy_01,The city should expand protected bike lanes be...,A local resident who supports environmental pr...,0.0,3,"""I'm aware of the proposal to expand protected...",llama3.1:latest,0.7,None
3,policy_01,The city should expand protected bike lanes be...,A local resident who supports environmental pr...,0.1,1,I'm cautiously in favor of expanding protected...,llama3.1:latest,0.7,None
4,policy_01,The city should expand protected bike lanes be...,A local resident who supports environmental pr...,0.1,2,While I appreciate the idea of expanding prote...,llama3.1:latest,0.7,None


In [29]:
JUDGE_PROMPT = """You are evaluating a generated social-agent message.
Judge only the text provided; do not assume the numeric Δ guarantees a particular result.

Return ONLY a valid JSON object with integer scores from 1 to 5:
{{
  "viewpoint_expression": 1,
  "viewpoint_fidelity": 1,
  "profile_consistency": 1,
  "verbatim_copying": 1,
  "brief_reason": "one short sentence"
}}

Scoring:
- viewpoint_expression: 1 = the original viewpoint is not expressed; 2 = barely implied; 3 = partially expressed; 4 = clearly expressed; 5 = very explicit and clear.
- viewpoint_fidelity: 1 = contradicts or substantially distorts the original viewpoint; 2 = weak fidelity; 3 = mixed/partial fidelity; 4 = mostly faithful; 5 = faithful to the core viewpoint without unsupported additions.
- profile_consistency: 1 = strongly inconsistent with the profile; 2 = mostly inconsistent; 3 = plausible but mixed; 4 = consistent; 5 = strongly consistent.
- verbatim_copying: 1 = no meaningful copying; 2 = minor phrase overlap; 3 = noticeable copying; 4 = substantial copying; 5 = nearly copied verbatim.

Do not score viewpoint expression based on message length alone.
A message can be faithful while expressing the viewpoint weakly.
A high Δ is not itself evidence that the message expresses the viewpoint clearly.

Original Post:
{post}

Target Profile:
{profile}

Δ supplied to generator: {delta}

Generated message:
{message}
"""


def extract_json(text):
    # Handle optional markdown fences and extra text around JSON.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in judge response: {text}")
    return json.loads(match.group(0))


def judge_message(post, profile, delta, message, temperature=0):
    prompt = JUDGE_PROMPT.format(
        post=post,
        profile=profile,
        delta=f"{float(delta):.2f}",
        message=message
    )
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=220
    )
    raw = response.choices[0].message.content.strip()
    return extract_json(raw), raw

In [ ]:
# Set to an integer (e.g., 10) to score only the first N rows during a pilot run.
PILOT_N = None

to_score = generations.dropna(subset=["generated_message"]).copy()
if PILOT_N is not None:
    to_score = to_score.head(PILOT_N)

score_rows = []
for idx, row in to_score.iterrows():
    try:
        scores, raw = judge_message(
            post=row["post"],
            profile=row["profile"],
            delta=row["delta"],
            message=row["generated_message"]
        )
        score_rows.append({
            "case_id": row["case_id"],
            "delta": row["delta"],
            "repeat": row["repeat"],
            "viewpoint_expression": scores["viewpoint_expression"],
            "viewpoint_fidelity": scores["viewpoint_fidelity"],
            "profile_consistency": scores["profile_consistency"],
            "verbatim_copying": scores["verbatim_copying"],
            "judge_reason": scores.get("brief_reason", ""),
            "judge_raw": raw,
            "judge_error": None
        })
    except Exception as e:
        score_rows.append({
            "case_id": row["case_id"],
            "delta": row["delta"],
            "repeat": row["repeat"],
            "viewpoint_expression": np.nan,
            "viewpoint_fidelity": np.nan,
            "profile_consistency": np.nan,
            "verbatim_copying": np.nan,
            "judge_reason": "",
            "judge_raw": "",
            "judge_error": str(e)
        })
        print("JUDGE ERROR:", row["case_id"], row["delta"], row["repeat"], e)

scores_df = pd.DataFrame(score_rows)
evaluation = generations.merge(scores_df, on=["case_id", "delta", "repeat"], how="left")
evaluation.to_csv("internalized_message_evaluation.csv", index=False, encoding="utf-8-sig")
evaluation.head()

In [ ]:
score_columns = [
    "viewpoint_expression",
    "viewpoint_fidelity",
    "profile_consistency",
    "verbatim_copying"
]

summary = (
    evaluation.groupby("delta")[score_columns]
    .agg(["mean", "std", "count"])
    .round(3)
)
display(summary)

means = evaluation.groupby("delta")[score_columns].mean()
ax = means[["viewpoint_expression", "viewpoint_fidelity", "profile_consistency"]].plot(
    marker="o", figsize=(9, 5)
)
ax.set_xlabel("Δ supplied to generator")
ax.set_ylabel("Mean judge score (1–5)")
ax.set_title("Message quality scores across Δ")
ax.set_ylim(1, 5)
ax.grid(True, alpha=0.3)
plt.show()

means[["verbatim_copying"]].plot(
    marker="o", figsize=(8, 4), legend=False, ylim=(1, 5),
    title="Verbatim copying across Δ"
)
plt.xlabel("Δ supplied to generator")
plt.ylabel("Mean copying score (1–5; lower is better)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def association_report(df):
    d = df.dropna(subset=["delta", "viewpoint_expression"])
    if d["delta"].nunique() < 2 or len(d) < 3:
        return {"n": len(d), "spearman_rho": np.nan, "spearman_p": np.nan,
                "kendall_tau": np.nan, "kendall_p": np.nan}
    rho, rho_p = spearmanr(d["delta"], d["viewpoint_expression"])
    tau, tau_p = kendalltau(d["delta"], d["viewpoint_expression"])
    return {
        "n": len(d),
        "spearman_rho": rho,
        "spearman_p": rho_p,
        "kendall_tau": tau,
        "kendall_p": tau_p
    }

print("Pooled:", association_report(evaluation))
print("\nBy case:")
for case_id, group in evaluation.groupby("case_id"):
    print(case_id, association_report(group))

# Mean expression score by case and Δ
case_means = evaluation.pivot_table(
    index="delta", columns="case_id", values="viewpoint_expression", aggfunc="mean"
)
display(case_means)
case_means.plot(marker="o", figsize=(9, 5))
plt.xlabel("Δ supplied to generator")
plt.ylabel("Mean viewpoint-expression score (1–5)")
plt.title("Viewpoint-expression trend by case")
plt.ylim(1, 5)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def show_case(case_id):
    cols = [
        "delta", "repeat", "generated_message",
        "viewpoint_expression", "viewpoint_fidelity",
        "profile_consistency", "verbatim_copying", "judge_reason"
    ]
    return (
        evaluation[evaluation["case_id"] == case_id][cols]
        .sort_values(["delta", "repeat"])
        .reset_index(drop=True)
    )

show_case(test_cases[0]["case_id"])

In [ ]:
from pathlib import Path
import pandas as pd

for filename in [
    "internalized_message_generations.csv",
    "internalized_message_evaluation.csv",
]:
    path = Path(filename)

    print(f"\n{'=' * 60}")
    print("File:", path.resolve())
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

In [ ]:
#New_run

In [ ]:
# If needed, uncomment:
# %pip install openai pandas numpy matplotlib scipy

from pathlib import Path
import json, re, time, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau
from openai import OpenAI

SEED = 20261023
np.random.seed(SEED)
DATA_DIR = Path.cwd()  # Change if your CSVs are in another folder
GENERATIONS_FILE = DATA_DIR / "internalized_message_generations.csv"
EVALUATIONS_FILE = DATA_DIR / "internalized_message_evaluation.csv"

MODEL = "llama3.1:latest"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

DELTAS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
PRIMARY_TEMPERATURE = 0.3
SENSITIVITY_TEMPERATURES = [0.1, 0.7]
PRIMARY_REPEATS = 3
SENSITIVITY_REPEATS = 3

print("Python:", platform.python_version())
print("Working directory:", DATA_DIR.resolve())
print("Model:", MODEL)


In [ ]:
def inspect_csv(path):
    print("\\n", "=" * 65, "\\nFile:", path.resolve(), "\\nExists:", path.exists())
    if not path.exists():
        return None
    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))
    return df

generations = inspect_csv(GENERATIONS_FILE)
evaluations_old = inspect_csv(EVALUATIONS_FILE)
if generations is None:
    raise FileNotFoundError(f"Move the generation CSV to {DATA_DIR} or update DATA_DIR.")

required = {"case_id", "post", "profile", "delta", "repeat", "generated_message"}
missing = required - set(generations.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

generations["delta"] = pd.to_numeric(generations["delta"], errors="coerce")
print("\\nGeneration errors:")
print(generations["error"].value_counts(dropna=False) if "error" in generations else "No error column")
display(generations.groupby("delta").size().rename("n_messages").to_frame())


In [ ]:
GENERATOR_PROMPT = """Generate a concise, natural message expressing the extent to which the viewpoint in the Original Post is reflected in the target user's response, consistent with the Target Profile and the given stance-change magnitude Δ.

Interpret Δ as a normalized control signal in [0, 1]:
- Δ near 0: express little or no clear reflection of the Original Post's viewpoint; the message may acknowledge the topic without clearly endorsing it.
- Moderate Δ: express a partial reflection of the viewpoint without fully endorsing it.
- Δ near 1: express the viewpoint more clearly and explicitly, while allowing reservations or selective agreement.

These are qualitative guidelines, not exact numerical mappings. Do not interpret Δ as a probability, percentage internalized, or guarantee of behavioral change. Stay consistent with the Target Profile. Do not introduce unrelated claims or copy the Original Post verbatim. Output only 1–3 sentences.

Original Post:
{post}

Target Profile:
{profile}

Δ:
{delta}

Generate only the message."""

JUDGE_PROMPT = """Evaluate the generated message. Return ONLY one valid JSON object:
{{
  "viewpoint_expression": 1,
  "viewpoint_fidelity": 1,
  "profile_consistency": 1,
  "verbatim_copying": 1,
  "brief_reason": "one short sentence"
}}

Use integer scores 1–5:
- viewpoint_expression: 1 = viewpoint not expressed; 3 = partly expressed; 5 = very clear and explicit.
- viewpoint_fidelity: 1 = contradicts/distorts the post; 3 = partly faithful; 5 = faithful without unsupported additions.
- profile_consistency: 1 = strongly inconsistent; 3 = mixed; 5 = strongly consistent.
- verbatim_copying: 1 = no meaningful copying; 3 = noticeable copying; 5 = nearly verbatim copying.

Do not judge expression by length alone. High Δ is not itself evidence of clear expression.

Original Post:
{post}

Target Profile:
{profile}

Δ supplied to generator: {delta}

Generated message:
{message}
"""

SCORE_COLS = ["viewpoint_expression", "viewpoint_fidelity", "profile_consistency", "verbatim_copying"]

def chat(prompt, temperature=0.0, max_tokens=400):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content or ""

def extract_json_object(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Empty judge response")
    cleaned = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", text.strip(), flags=re.I)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError(f"No JSON object found: {cleaned[:160]!r}")
    depth, in_string, escaped = 0, False, False
    for i in range(start, len(cleaned)):
        ch = cleaned[i]
        if in_string:
            if escaped: escaped = False
            elif ch == "\\\\": escaped = True
            elif ch == '"': in_string = False
        else:
            if ch == '"': in_string = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return json.loads(cleaned[start:i+1])
    raise ValueError("Incomplete JSON object")

def judge_message(post, profile, delta, message):
    prompt = JUDGE_PROMPT.format(post=post, profile=profile, delta=float(delta), message=message)
    raw = chat(prompt, temperature=0.0, max_tokens=300)
    parsed = extract_json_object(raw)
    out = {}
    for col in SCORE_COLS:
        val = pd.to_numeric(parsed.get(col), errors="coerce")
        if pd.isna(val) or int(val) != val or not 1 <= int(val) <= 5:
            raise ValueError(f"Invalid {col} score: {parsed.get(col)!r}")
        out[col] = int(val)
    out["judge_reason"] = str(parsed.get("brief_reason", ""))
    out["judge_raw"] = raw
    out["judge_error"] = np.nan
    return out

# Smoke test: make one judge call before scoring all rows.
s = generations.iloc[0]
test = judge_message(s["post"], s["profile"], s["delta"], s["generated_message"])
print({k: test[k] for k in SCORE_COLS}, test["judge_reason"])


In [ ]:
def score_dataframe(df):
    rows = []
    for i, row in df.reset_index(drop=True).iterrows():
        result = row.to_dict()
        try:
            result.update(judge_message(row["post"], row["profile"], row["delta"], row["generated_message"]))
        except Exception as exc:
            result.update({c: np.nan for c in SCORE_COLS})
            result["judge_reason"] = ""
            result["judge_raw"] = ""
            result["judge_error"] = repr(exc)
        rows.append(result)
        if (i + 1) % 10 == 0 or i + 1 == len(df):
            print(f"Scored {i+1}/{len(df)}")
    return pd.DataFrame(rows)

RERUN_EXISTING_JUDGE = True
if RERUN_EXISTING_JUDGE:
    if EVALUATIONS_FILE.exists():
        backup = EVALUATIONS_FILE.with_name(
            f"internalized_message_evaluation_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        )
        evaluations_old.to_csv(backup, index=False)
        print("Backup:", backup.resolve())
    existing_scored = score_dataframe(generations)
    existing_scored.to_csv(EVALUATIONS_FILE, index=False)
else:
    existing_scored = pd.read_csv(EVALUATIONS_FILE)

print("Valid numeric scores:")
display(existing_scored[SCORE_COLS].apply(pd.to_numeric, errors="coerce").notna().sum())
print("Judge errors:")
display(existing_scored["judge_error"].value_counts(dropna=False).head(10))


In [ ]:
def summarize(df, group_cols="delta"):
    d = df.copy()
    for c in SCORE_COLS:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d.groupby(group_cols)[SCORE_COLS].agg(["mean", "std", "count"]).round(3)

def plot_means(df, title):
    means = df.groupby("delta")[SCORE_COLS].mean(numeric_only=True)
    ax = means.plot(marker="o", figsize=(9, 5))
    ax.set_xlabel("delta")
    ax.set_ylabel("Mean judge score (1–6)")
    ax.set_ylim(1, 7)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

display(summarize(existing_scored))
plot_means(existing_scored, "Pilot: mean judge scores by delta")


In [ ]:
# These are illustrative test cases. Edit/add cases to reflect your intended domain.
scenario_rows = [
("bike_01", "The city should expand protected bike lanes because they can make cycling safer and reduce dependence on cars.", "A local resident who supports environmental protection but worries about road congestion and parking."),
("remote_02", "Companies should offer hybrid work because it can improve flexibility while preserving in-person collaboration.", "An employee who values flexibility but believes mentoring and teamwork are easier in person."),
("school_03", "Schools should start later because additional sleep can support student well-being and learning.", "A parent who cares about student health but is concerned about transport schedules and after-school activities."),
("renewable_04", "The region should invest more in renewable energy to reduce emissions and improve long-term energy security.", "A resident who supports cleaner energy but worries about reliability and household costs."),
("library_05", "Public libraries should extend evening hours to make educational resources accessible to more residents.", "A community member who values public education but is concerned about staffing and operating budgets."),
("smallbiz_06", "Local government should simplify permits for small businesses to encourage neighborhood employment and growth.", "A resident who wants local businesses to succeed but also values safety rules and fair competition."),
("recycling_07", "The city should expand household recycling services to reduce waste and conserve resources.", "A resident who supports reducing waste but doubts whether collection and sorting systems work efficiently."),
("transit_08", "Public transit should receive more funding because reliable service can reduce traffic and improve access to jobs.", "A commuter who wants less traffic but is concerned about taxes and current service quality."),
]
scenarios = pd.DataFrame(scenario_rows, columns=["case_id", "post", "profile"])
SCENARIO_FILE = DATA_DIR / "calibration_scenarios.csv"
scenarios.to_csv(SCENARIO_FILE, index=False)
display(scenarios)
print("Saved:", SCENARIO_FILE.resolve())


In [ ]:
PRIMARY_GENERATIONS_FILE = DATA_DIR / "expanded_primary_generations.csv"
RUN_EXPANDED_PRIMARY = True  # Set True when ready.

def load_or_empty(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def run_generation_design(cases, deltas, repeats, temperature, output_path):
    old = load_or_empty(output_path)
    records = old.to_dict("records") if not old.empty else []
    seen = set()
    if not old.empty:
        for _, r in old.iterrows():
            seen.add((
    str(r["case_id"]),
    float(r["delta"]),
    int(r["repeat"]),
    float(r["temperature"]),
    str(r["generation_model"])
))

    tasks = []
    for _, case in cases.iterrows():
        for delta in deltas:
            for rep in range(1, repeats + 1):
                key = (str(case.case_id), float(delta), rep, float(temperature), MODEL)
                if key not in seen:
                    tasks.append((case, float(delta), rep))
    print("New generation calls:", len(tasks))

    for i, (case, delta, rep) in enumerate(tasks, 1):
        row = {
            "case_id": case.case_id, "post": case.post, "profile": case.profile,
            "delta": delta, "repeat": rep, "generation_model": MODEL,
            "temperature": float(temperature), "prompt_version": "qualitative_delta_anchors_v1",
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "generated_message": "", "error": np.nan
        }
        try:
            row["generated_message"] = chat(
                GENERATOR_PROMPT.format(post=case.post, profile=case.profile, delta=delta),
                temperature=temperature, max_tokens=220
            ).strip()
        except Exception as exc:
            row["error"] = repr(exc)
        records.append(row)
        pd.DataFrame(records).to_csv(output_path, index=False)  # save progress for resume
        if i % 10 == 0 or i == len(tasks):
            print(f"Completed {i}/{len(tasks)} new calls")
    return pd.DataFrame(records)

if RUN_EXPANDED_PRIMARY:
    expanded_primary = run_generation_design(
        scenarios, DELTAS, PRIMARY_REPEATS, PRIMARY_TEMPERATURE, PRIMARY_GENERATIONS_FILE
    )
else:
    expanded_primary = load_or_empty(PRIMARY_GENERATIONS_FILE)
    print("Not run. Set RUN_EXPANDED_PRIMARY=True to start.")


In [ ]:
SENSITIVITY_GENERATIONS_FILE = DATA_DIR / "temperature_sensitivity_generations.csv"
RUN_TEMPERATURE_SENSITIVITY = True  # Set True when ready.

if RUN_TEMPERATURE_SENSITIVITY:
    sensitivity_parts = []
    for temp in SENSITIVITY_TEMPERATURES:
        path = DATA_DIR / f"sensitivity_temp_{str(temp).replace('.', '_')}.csv"
        part = run_generation_design(
            scenarios.iloc[:4].copy(), DELTAS, SENSITIVITY_REPEATS, temp, path
        )
        sensitivity_parts.append(part)
    sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
    sensitivity.to_csv(SENSITIVITY_GENERATIONS_FILE, index=False)
else:
    sensitivity = load_or_empty(SENSITIVITY_GENERATIONS_FILE)
    print("Not run. Set RUN_TEMPERATURE_SENSITIVITY=True to start.")


In [ ]:
def score_file(input_path, output_path):
    df = pd.read_csv(input_path)
    eligible = df[df["generated_message"].notna() & df["generated_message"].astype(str).str.strip().ne("")].copy()
    if "error" in eligible:
        eligible = eligible[eligible["error"].isna()]
    old = load_or_empty(output_path)
    records = old.to_dict("records") if not old.empty else []
    seen = set()
    if not old.empty:
        for _, r in old.iterrows():
            seen.add((str(r.case_id), float(r.delta), int(r.repeat), float(r.temperature), str(r.generation_model)))
    tasks = []
    for _, r in eligible.iterrows():
        key = (
    str(r["case_id"]),
    float(r["delta"]),
    int(r["repeat"]),
    float(r["temperature"]),
    str(r["generation_model"])
)
        if key not in seen:
            tasks.append(r)
    print("New judge calls:", len(tasks))
    for i, r in enumerate(tasks, 1):
        out = r.to_dict()
        try:
            out.update(judge_message(r.post, r.profile, r.delta, r.generated_message))
        except Exception as exc:
            out.update({c: np.nan for c in SCORE_COLS})
            out["judge_reason"], out["judge_raw"], out["judge_error"] = "", "", repr(exc)
        records.append(out)
        pd.DataFrame(records).to_csv(output_path, index=False)
        if i % 10 == 0 or i == len(tasks):
            print(f"Scored {i}/{len(tasks)}")
    return pd.DataFrame(records)

In [ ]:

RUN_SCORING = True  # Set True after generation files exist.
if RUN_SCORING:
    if PRIMARY_GENERATIONS_FILE.exists():
        primary_scored = score_file(PRIMARY_GENERATIONS_FILE, DATA_DIR / "expanded_primary_evaluation.csv")
    if SENSITIVITY_GENERATIONS_FILE.exists():
        sensitivity_scored = score_file(SENSITIVITY_GENERATIONS_FILE, DATA_DIR / "temperature_sensitivity_evaluation.csv")
else:
    print("Not run. Set RUN_SCORING=True after generating messages.")


In [ ]:
def trend_report(df):
    d = df.copy()
    for c in SCORE_COLS + ["delta"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    valid = d.dropna(subset=["delta", "viewpoint_expression"])
    rho, p = spearmanr(valid["delta"], valid["viewpoint_expression"])
    tau, p_tau = kendalltau(valid["delta"], valid["viewpoint_expression"])
    print(f"Message-level Spearman rho={rho:.3f}, p={p:.4g}")
    print(f"Message-level Kendall tau={tau:.3f}, p={p_tau:.4g}")
    print("Caution: these p-values do not account for repeated observations within cases.")
    print("\\nOverall summary:")
    display(summarize(valid))
    print("\\nCase-by-delta means (inspect whether the trend generalizes across cases):")
    display(valid.groupby(["case_id", "delta"])[SCORE_COLS].mean().round(2))

# After running scoring
trend_report(primary_scored)
plot_means(primary_scored, "Expanded primary calibration")
display(sensitivity_scored.groupby(["temperature", "delta"])[SCORE_COLS].mean().round(2))


In [ ]:
def export_human_review_sheet(df, n_per_delta=4, seed=SEED):
    d = df[df["generated_message"].notna() & df["generated_message"].astype(str).str.strip().ne("")].copy()
    selected = []
    rng = np.random.default_rng(seed)
    for delta, group in d.groupby("delta"):
        n = min(n_per_delta, len(group))
        selected.append(d.loc[rng.choice(group.index.to_numpy(), n, replace=False)])
    sample = pd.concat(selected, ignore_index=True)
    sample.insert(0, "review_id", [f"R{i:03d}" for i in range(1, len(sample)+1)])
    review = sample[["review_id", "post", "profile", "generated_message"]].copy()
    for c in ["human_viewpoint_expression", "human_viewpoint_fidelity", "human_profile_consistency", "human_verbatim_copying", "human_notes"]:
        review[c] = ""
    key_cols = ["review_id", "case_id", "delta", "repeat", "temperature"] + SCORE_COLS
    key = sample[[c for c in key_cols if c in sample.columns]].copy()
    review_path = DATA_DIR / "human_review_blinded.csv"
    key_path = DATA_DIR / "human_review_key.csv"
    review.to_csv(review_path, index=False)
    key.to_csv(key_path, index=False)
    print("Give reviewer this file:", review_path.resolve())
    print("Keep this key file hidden:", key_path.resolve())
    return review, key

# Example:
human_sheet, human_key = export_human_review_sheet(primary_scored, n_per_delta=4)


In [ ]:
def compare_human_and_judge(completed_review_path, key_path):
    human = pd.read_csv(completed_review_path)
    key = pd.read_csv(key_path)
    merged = human.merge(key, on="review_id", how="inner")
    human_cols = {
        "viewpoint_expression": "human_viewpoint_expression",
        "viewpoint_fidelity": "human_viewpoint_fidelity",
        "profile_consistency": "human_profile_consistency",
        "verbatim_copying": "human_verbatim_copying",
    }
    rows = []
    for judge_col, human_col in human_cols.items():
        x = pd.to_numeric(merged[human_col], errors="coerce")
        y = pd.to_numeric(merged[judge_col], errors="coerce")
        ok = x.notna() & y.notna()
        if ok.sum() >= 3:
            rho, _ = spearmanr(x[ok], y[ok])
            mae = np.mean(np.abs(x[ok] - y[ok]))
        else:
            rho, mae = np.nan, np.nan
        rows.append({"metric": judge_col, "n": int(ok.sum()), "spearman_rho": rho, "MAE": mae})
    return pd.DataFrame(rows)

# After rating the CSV, save it as human_review_blinded_completed.csv:
display(compare_human_and_judge(DATA_DIR / "human_review_blinded_completed.csv", DATA_DIR / "human_review_key.csv"))
